# Solar filament segmentation -- Colab training run `c1-r34-1024`

One-time setup (about 2 minutes):

1. **Runtime -> Change runtime type -> T4 GPU.**
2. On Kaggle: *Settings -> API Tokens -> Generate New Token* and copy it.
   (Do **not** use "Create Legacy API Key": that would expire the key already in use.)
3. In Colab, open the key icon (Secrets) in the left sidebar and add two secrets with
   *Notebook access* switched on:
   * `KAGGLE_API_TOKEN` = the token from step 2
   * `KAGGLE_USERNAME` = your Kaggle user name
4. **Runtime -> Run all.**

If the session disconnects, reconnect and *Run all* again: training resumes from the
last checkpoint, which is stored in the private Kaggle dataset `fil-colab-c1-r34-1024`.
Results (weights, logs, validation scores, submission) are uploaded to the same
private dataset automatically; nothing needs to be downloaded by hand.


In [ ]:
RUN = 'c1-r34-1024'
TRAIN_ARGS = ['--res', '1024', '--encoder', 'resnet34', '--epochs', '40', '--bs', '6', '--fold', '0', '--val-every', '5', '--tta', '4', '--workers', '2', '--ema', '0.999', '--max-hours', '3.3', '--predict-test']
SYNC_MINUTES = 20  # upload the checkpoint at most this often


In [ ]:
import os, subprocess, sys, json, time, shutil, threading, base64, io, zipfile, glob
from pathlib import Path
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "kaggle", "segmentation-models-pytorch", "pycocotools"], check=True)
from google.colab import userdata
os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
USER = userdata.get("KAGGLE_USERNAME").strip()
SLUG = f"{USER}/fil-colab-{RUN}"
WORK = Path("/content/work"); OUT = WORK / "out" / RUN; OUT.mkdir(parents=True, exist_ok=True)
def kaggle(*a, check=True):
    r = subprocess.run(["kaggle", *a], capture_output=True, text=True)
    if check and r.returncode:
        raise RuntimeError(r.stdout + r.stderr)
    return r
print("kaggle user:", USER)


In [ ]:
DATA = WORK / "data"
if not (DATA / "MAGFiLO_1.0_Kaggle_2026").exists():
    kaggle("competitions", "download", "filament-segmentation-2026", "-p", str(DATA), "-q")
    for z in DATA.glob("*.zip"):
        zipfile.ZipFile(z).extractall(DATA); z.unlink()
print(sorted(p.name for p in (DATA / "MAGFiLO_1.0_Kaggle_2026").iterdir()))


In [ ]:
PAYLOAD = "..."
SRC = WORK / "src"; shutil.rmtree(SRC, ignore_errors=True)
zipfile.ZipFile(io.BytesIO(base64.b64decode(PAYLOAD))).extractall(SRC)
print("code unpacked to", SRC)


In [ ]:
SYNC = WORK / "sync"
def dataset_exists():
    return kaggle("datasets", "status", SLUG, check=False).returncode == 0
if not (OUT / "last.pt").exists() and dataset_exists():
    tmp = WORK / "resume"; shutil.rmtree(tmp, ignore_errors=True)
    r = kaggle("datasets", "download", SLUG, "-p", str(tmp), "--unzip", check=False)
    for f in ["last.pt", "log.txt"]:
        if (tmp / f).exists():
            shutil.copy(tmp / f, OUT / f)
    print("resume files:", sorted(p.name for p in OUT.iterdir()))


In [ ]:
_lock = threading.Lock()
def sync(message, files):
    with _lock:
        shutil.rmtree(SYNC, ignore_errors=True); SYNC.mkdir(parents=True)
        for f in files:
            if (OUT / f).exists():
                shutil.copy(OUT / f, SYNC / f)
        (SYNC / "dataset-metadata.json").write_text(json.dumps(
            {"title": f"fil-colab-{RUN}", "id": SLUG, "licenses": [{"name": "other"}]}))
        if dataset_exists():
            r = kaggle("datasets", "version", "-p", str(SYNC), "-m", message, "-q", check=False)
        else:
            r = kaggle("datasets", "create", "-p", str(SYNC), "-q", check=False)  # private by default
        print(f"[sync] {message}: rc={r.returncode} {r.stderr.strip()[-200:]}", flush=True)

stop = threading.Event()
def periodic():
    last = None
    while not stop.wait(SYNC_MINUTES * 60):
        p = OUT / "last.pt"
        if p.exists() and p.stat().st_mtime != last:
            last = p.stat().st_mtime
            sync(f"checkpoint {time.strftime('%H:%M')}", ["last.pt", "log.txt"])
threading.Thread(target=periodic, daemon=True).start()

env = dict(os.environ, PYTHONPATH=str(SRC), PYTHONUNBUFFERED="1")
cmd = [sys.executable, "-m", "filseg.train", "--data", str(DATA / "MAGFiLO_1.0_Kaggle_2026"), "--out", str(OUT), *TRAIN_ARGS]
proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
rc = proc.wait(); stop.set()
print("training exit code", rc)
sync(f"final rc={rc}", ["model.pt", "post.json", "log.txt", "val_probs.npz", "submission.csv"])
